# Study & Marketing Assistant — Practical Assistant Agent
**Work 2 | Student project**

This agent helps students manage tasks, check current weather in Tbilisi, and find books for study topics. It uses Groq as its LLM, LangGraph for the workflow and memory, Open-Meteo for weather, and Open Library for book search.

## Setup
1. Save a copy of this notebook to your Google Drive.
2. In Colab → Secrets, create `GROQ_API_KEY`, paste your own Groq key, and enable Notebook access.
3. Run the cells from top to bottom. Never print or upload your API key.


In [ ]:
%pip install -q langgraph langchain-groq langchain-core requests


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
assert os.environ.get("GROQ_API_KEY"), "Add GROQ_API_KEY in Colab Secrets first."
print("API key loaded.")


## Project description
**Study & Marketing Assistant** is an AI assistant for students. It helps users organize academic and personal tasks, find books by topic, and check current weather in Tbilisi. It remembers the conversation during the current session and asks for explicit approval before deleting a task.


## Build the agent
The reusable implementation is stored in `agent.py`. In Colab, the next cell writes that file. If you are working from the provided project ZIP, you can instead upload `agent.py` to the Colab session.


In [ ]:
%%writefile agent.py
"""Practical Assistant Agent: LangGraph + Groq + external APIs."""
import os
import requests
from typing import Annotated, TypedDict
from langchain_groq import ChatGroq
from langchain_core.messages import AnyMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]


SYSTEM_PROMPT = """You are Study & Marketing Assistant, a helpful assistant for students.
You can manage a task list, check current weather in Tbilisi, and search for books.
Use tools when the user asks for these actions. Answer in the language used by the user.
Never claim an action succeeded unless the relevant tool confirms it.
Before deleting a task, the delete tool will request explicit approval. If the user
does not approve, do not try another way to delete it. Keep answers clear and concise."""


def build_graph(task_store=None, model_name="llama-3.3-70b-versatile"):
    """Build and compile one isolated assistant graph."""
    tasks = task_store if task_store is not None else []

    @tool
    def add_task(task: str) -> str:
        """Add a new task to the user's task list."""
        task = task.strip()
        if not task:
            return "Task was not added because the task description was empty."
        tasks.append(task)
        return f"Task added: {task}"

    @tool
    def list_tasks() -> str:
        """Show all tasks currently saved in the task list."""
        if not tasks:
            return "Your task list is empty."
        return "\n".join(f"{i + 1}. {task}" for i, task in enumerate(tasks))

    @tool
    def delete_task(task: str) -> str:
        """Delete a task only after the user explicitly approves the exact deletion."""
        matches = [item for item in tasks if item.casefold() == task.strip().casefold()]
        if not matches:
            return f"No matching task found: {task}"
        approval = interrupt({
            "action": "delete_task",
            "task": matches[0],
            "message": f"Please confirm: delete the task '{matches[0]}'?"
        })
        if approval is not True:
            return f"Deletion cancelled. The task remains: {matches[0]}"
        tasks.remove(matches[0])
        return f"Task deleted: {matches[0]}"

    @tool
    def get_tbilisi_weather() -> str:
        """Get the current temperature and weather information in Tbilisi, in Celsius."""
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": 41.72,
                "longitude": 44.79,
                "current": "temperature_2m,relative_humidity_2m,weather_code",
            },
            timeout=20,
        )
        response.raise_for_status()
        current = response.json()["current"]
        return (
            f"Tbilisi current weather: {current.get('temperature_2m')}°C; "
            f"humidity: {current.get('relative_humidity_2m')}%; "
            f"weather code: {current.get('weather_code')}."
        )

    @tool
    def search_books(topic: str) -> str:
        """Search Open Library for a few books on a topic and return titles and authors."""
        response = requests.get(
            "https://openlibrary.org/search.json",
            params={"q": topic, "limit": 5, "fields": "title,author_name,first_publish_year"},
            timeout=20,
        )
        response.raise_for_status()
        docs = response.json().get("docs", [])
        if not docs:
            return f"No books found for: {topic}"
        lines = []
        for item in docs[:5]:
            title = item.get("title", "Unknown title")
            authors = ", ".join(item.get("author_name", [])[:2]) or "Unknown author"
            year = item.get("first_publish_year")
            suffix = f" ({year})" if year else ""
            lines.append(f"- {title} — {authors}{suffix}")
        return "\n".join(lines)

    tools = [add_task, list_tasks, delete_task, get_tbilisi_weather, search_books]
    if not os.getenv("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY is missing. Add it to your environment or Colab Secrets first."
        )

    llm = ChatGroq(
        model=model_name,
        temperature=0,
        max_tokens=700,
    )
    llm_with_tools = llm.bind_tools(tools)

    def assistant(state: AgentState):
        messages = state["messages"]
        if not messages or not isinstance(messages[0], SystemMessage):
            messages = [SystemMessage(content=SYSTEM_PROMPT)] + messages
        response = llm_with_tools.invoke(messages)
        return {"messages": [response]}

    def route_after_assistant(state: AgentState):
        last = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else END

    builder = StateGraph(AgentState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", route_after_assistant, ["tools", END])
    builder.add_edge("tools", "assistant")

    checkpointer = InMemorySaver()
    graph = builder.compile(checkpointer=checkpointer)
    return graph, checkpointer, tasks


def new_config(thread_id="study-assistant-1"):
    """Return a LangGraph config for a persistent conversation thread."""
    return {"configurable": {"thread_id": thread_id}}


def ask(graph, user_text, config):
    """Send a message to the agent. A delete request may pause for approval."""
    from langchain_core.messages import HumanMessage
    return graph.invoke({"messages": [HumanMessage(content=user_text)]}, config)


def resume(graph, config, approved: bool):
    """Resume a paused action with True (approve) or False (deny)."""
    return graph.invoke(Command(resume=bool(approved)), config)


In [ ]:
from agent import build_graph, new_config, ask, resume
from langchain_core.messages import HumanMessage

graph, checkpointer, tasks = build_graph()
config = new_config("colab-study-session")
print("Agent graph is ready.")


## Test 1 — Add a task
Expected result: the agent calls `add_task` and confirms that the task was added.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Add task: prepare my CRM presentation")]}, config)
for m in result["messages"]:
    m.pretty_print()


## Test 2 — List tasks / memory
Expected result: the task added in Test 1 appears in the list.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Show my tasks")]}, config)
for m in result["messages"]:
    m.pretty_print()


## Test 3 — Weather API
Expected result: the agent calls Open-Meteo and returns current Tbilisi weather. This test needs internet access.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="What is the current temperature in Tbilisi?")]}, config)
for m in result["messages"]:
    m.pretty_print()


## Test 4 — Book search API
Expected result: the agent searches Open Library and returns several book titles and authors.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Find books about Python programming")]}, config)
for m in result["messages"]:
    m.pretty_print()


## Test 5 — Delete task, approved
Expected result: the agent pauses and displays the exact task to delete. After approval, the task is deleted.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Delete the task prepare my CRM presentation")]}, config)
pending = result.get("__interrupt__", [])
print("Approval request:", pending[0].value if pending else "No approval was requested")


In [ ]:
# Run this cell only after checking the approval request above.
from langgraph.types import Command
result = graph.invoke(Command(resume=True), config)
for m in result["messages"]:
    m.pretty_print()


## Test 6 — Delete task, denied
Create a separate task, request deletion, then deny it. Expected result: the task remains in the list.


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Add task: read marketing chapter")]}, config)
for m in result["messages"]:
    m.pretty_print()


In [ ]:
result = graph.invoke({"messages": [HumanMessage(content="Delete the task read marketing chapter")]}, config)
pending = result.get("__interrupt__", [])
print("Approval request:", pending[0].value if pending else "No approval was requested")


In [ ]:
# Deny the deletion. The task should remain unchanged.
result = graph.invoke(Command(resume=False), config)
for m in result["messages"]:
    m.pretty_print()


## Evaluation table
Fill the Pass? and Notes columns after running each test. Do not mark a test as passed unless the actual output matches the expected result.


| # | Query | Expected Result | Pass? | Notes |
|---|---|---|---|---|
| 1 | Add task | Task added | ☐ | Complete after running Test 1 |
| 2 | Show my tasks | Previously added task appears | ☐ | Checks memory and task storage |
| 3 | Tbilisi weather | Current weather returned by API | ☐ | Requires internet |
| 4 | Search Python books | Titles and authors returned | ☐ | Uses Open Library |
| 5 | Delete task — approve | Approval requested, then task deleted | ☐ | Confirm only after checking task name |
| 6 | Delete task — deny | Approval requested, task remains | ☐ | Confirms safe behavior |


## StateGraph diagram
The graph routes each user message to the assistant. If the model requests a tool, the ToolNode runs it and returns the result to the assistant. Otherwise, the assistant returns its final answer.


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## Notes and limitations
- `InMemorySaver` keeps memory only during the current running session; restarting the runtime clears it.
- The task list is stored in memory and is not a permanent database.
- Weather and book search require internet access.
- Groq has rate limits; if a 429 error appears, wait and retry.
- Never commit your real API key to GitHub.
